# EDA: modelarea si explorarea datelor (predare)

Apasa **Copy to Drive** (sus, in stanga) inainte sa scrii ceva aici, altfel tot ce faci dispare cand inchizi tab-ul.

Ruleaza prima celula de mai jos, o singura data, ca sa aduci utilitarele cursului.

In [ ]:
import os, urllib.request
if not os.path.exists('onia.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/alexbirs27/iLoveBeer/main/cursuri/colab/onia.py', 'onia.py')
from onia import *
descarca('date/elevi.csv', 'date/case.csv', 'anim_eda.py')
from anim_eda import *

## Ce e EDA si de ce o faci inaintea oricarui model

EDA (Exploratory Data Analysis) inseamna sa te uiti la date **inainte** sa le dai unui model. Nu e o formalitate: majoritatea greselilor de la concurs nu sunt in model, sunt in date. O varsta de 150 de ani, o coloana citita ca text, acelasi raspuns scris in trei feluri, cateva randuri copiate de doua ori.

Modelul nu se plange de nimic din toate astea. Iti da doar un scor mai prost, iar tu crezi ca ai ales algoritmul gresit.

In acest notebook:

1. ce inseamna fiecare coloana (tipuri de feature-uri);
2. statistici de baza **de mana** (medie, mediana, varianta, cuantile, IQR), apoi cu pandas, apoi de ce pandas e mai rapid;
3. selectii, sortari, grupari;
4. grafice: distributii, outlieri, relatii;
5. o lista de intrebari (checklist) pe care o pui oricarui set de date. O aplicam pe `date/elevi.csv`, care e murdar intentionat, si apoi pe `date/case.csv`.

Pandas il invatam pe parcurs, doar cat ai nevoie. **Nu memora biblioteca**: la olimpiada ai documentatia. Trebuie sa stii ce *intrebare* pui datelor si sa poti scrie de mana ce face functia.

**Cum lucram.** Celulele `%%vezi` executa cod Python simplu pas cu pas (liste, bucle, dictionare). Nu pot afisa tabele pandas, asa ca partea "de mana" o vezi acolo, iar partea cu tabele o vezi in grafice si in animatii facute pe datele tale.

## 1. Ce e o coloana: tipurile de feature-uri

Inainte de orice calcul te intrebi: ce fel de coloana am in fata? Asta decide ce ai voie sa faci cu ea (nu are sens media la `oras`).

| tip | exemplu in `elevi.csv` | ce faci cu el |
|---|---|---|
| numeric continuu | `nota_mate`, `ore_studiu` | medie, histograma, corelatie |
| numeric discret (numarat) | `absente` | la fel, dar valori intregi |
| categoric (nominal) | `meditatii` (da / nu) | `value_counts`, `groupby` |
| ordinal (are ordine) | `clasa` (9 < 10 < 11 < 12) | il poti trata ca numar sau ca categorie |
| **tinta** (target) | `calificat` (0 / 1) | ce vrem sa prezicem |
| identificator | `id` | **nu e feature**: nu-l dai modelului |

Un identificator nu spune nimic despre elev, dar daca randurile sunt ordonate dupa ceva, modelul poate "invata" ordinea. Il scoti.

Incarcam datele. Fiecare rand e un elev, iar `calificat` spune daca a trecut mai departe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

elevi = pd.read_csv('date/elevi.csv')
elevi.head()

## 2. Media si mediana, de mana

**Media** = suma / cate sunt. **Mediana** = valoarea din mijloc, dupa ce sortezi lista. Uite-le pas cu pas:

In [ ]:
%%vezi
note = [7, 9, 5, 8, 6]
suma = 0
for x in note:
    suma = suma + x       #: adunam pe rand
media = suma / len(note)  #: suma impartita la cate sunt

**Prezice:** ai lista `[3, 4, 5, 5, 6, 100]`. Care e mai mare, media sau mediana, si aproximativ cu cat?

<details><summary>Raspuns</summary>

Media e `123 / 6 = 20.5`, mediana e `5`. O singura valoare uriasa trage media dupa ea, iar mediana nici nu observa: ea se uita doar la ce e in mijloc. De aici regula: **la date cu valori extreme sau asimetrice, mediana e mai de incredere**.
</details>

In [ ]:
%%vezi
v = [3, 9, 4, 100, 5, 5]
s = sorted(v)                              #: mediana cere lista sortata
n = len(s)
mij = n // 2
if n % 2 == 1:
    mediana = s[mij]                       #: numar impar: elementul din mijloc
else:
    mediana = (s[mij - 1] + s[mij]) / 2    #: numar par: media celor doua din mijloc
media = sum(v) / n

## 3. Varianta si deviatia standard, de mana

Media spune *unde* e centrul. Varianta spune *cat de imprastiate* sunt valorile in jurul lui: media patratelor distantelor la medie. Deviatia standard e radacina ei, ca sa revina la aceeasi unitate ca datele.

Doua formule, si aici se incurca toata lumea:

- **populatie** (ai TOATE datele): imparti la `n`
- **esantion** (ai doar o parte din date, cazul obisnuit): imparti la `n - 1`. Se numeste `ddof=1` ("delta degrees of freedom"). Motivul pe scurt: distantele se masoara fata de media *esantionului*, care e mai aproape de puncte decat media adevarata, deci `n` ar da un rezultat mereu prea mic.

In [ ]:
%%vezi
x = [2, 4, 4, 4, 5, 5, 7, 9]
n = len(x)
m = sum(x) / n                    #: media este 5
patrate = 0
for v in x:
    patrate += (v - m) ** 2       #: cat de departe e fiecare de medie, la patrat
var_pop = patrate / n             #: impartit la n: populatie
var_esantion = patrate / (n - 1)  #: impartit la n-1: esantion
std_pop = var_pop ** 0.5

**Prezice:** pentru aceeasi lista, ce iti dau `np.std(x)` si `pd.Series(x).std()`? Sunt egale?

<details><summary>Raspuns</summary>

Nu. `np.std` are implicit `ddof=0` (formula de populatie), iar pandas are implicit `ddof=1` (esantion). Aceeasi lista, doua rezultate. Cand compari cu un rezultat asteptat la concurs, uita-te ce formula cere enuntul.
</details>

In [ ]:
x = [2, 4, 4, 4, 5, 5, 7, 9]
print('numpy :', np.std(x))
print('pandas:', pd.Series(x).std())
print('numpy cu ddof=1:', np.std(x, ddof=1))
print('pandas cu ddof=0:', pd.Series(x).std(ddof=0))

## 4. Cuantile si IQR, de mana

Cuantila `q` (de ex. 0.25) e valoarea sub care se afla o fractiune `q` din date. Cele trei importante: `Q1` (0.25), mediana (0.5), `Q3` (0.75). **IQR = Q3 - Q1** e "latimea mijlocului" datelor.

Cum o calculezi: sortezi, iei pozitia `q * (n - 1)`, iar daca pozitia cade intre doua elemente, interpolezi liniar intre ele.

Regula pentru outlieri (Tukey): o valoare e suspecta daca iese din **garduri**:

`gard_jos = Q1 - 1.5 * IQR`, `gard_sus = Q3 + 1.5 * IQR`

In [ ]:
%%vezi
def cuantila(s, q):
    poz = q * (len(s) - 1)     #: pozitia in lista sortata, poate cu virgula
    jos = int(poz)
    frac = poz - jos
    if jos + 1 < len(s):
        return s[jos] + frac * (s[jos + 1] - s[jos])   #: interpolam intre vecini
    return s[jos]

s = sorted([2, 3, 4, 6, 8, 9, 10, 40])
q1 = cuantila(s, 0.25)
q3 = cuantila(s, 0.75)
iqr = q3 - q1
gard_jos = q1 - 1.5 * iqr
gard_sus = q3 + 1.5 * iqr
outlieri = [x for x in s if x < gard_jos or x > gard_sus]   #: ce iese din garduri

NumPy face exact asta (`np.percentile`, interpolare liniara), la fel `Series.quantile`. Atentie: exista mai multe definitii de cuantila (manualele nu se pun de acord la `Q1`). Pandas si NumPy o folosesc pe cea de mai sus, deci rezultatele lor sunt cele "oficiale" pentru noi.

In [ ]:
s = [2, 3, 4, 6, 8, 9, 10, 40]
print(np.percentile(s, [25, 75]))
print(pd.Series(s).quantile([0.25, 0.75]).tolist())

## 5. De ce pandas / NumPy e mai rapid

Ai scris de mana media cu o bucla. Functia din librarie face acelasi lucru, dar:

- valorile stau intr-un **bloc continuu de memorie**, cate 8 octeti fiecare, nu ca obiecte Python separate;
- bucla ruleaza in **C**, nu in interpretorul Python (care verifica tipul fiecarui `+`);
- procesorul poate aduna mai multe numere deodata.

Masoara:

In [ ]:
import time

mare = np.random.default_rng(0).random(2_000_000)
lista = mare.tolist()

t = time.perf_counter()
suma = 0.0
for x in lista:
    suma += x
t_bucla = time.perf_counter() - t

t = time.perf_counter()
suma_np = mare.sum()
t_np = time.perf_counter() - t

print(f'bucla Python: {t_bucla * 1000:.0f} ms | NumPy: {t_np * 1000:.1f} ms | de ~{t_bucla / t_np:.0f} ori mai rapid')
print(abs(suma - suma_np) < 1e-6)

Un `Series` din pandas e un array NumPy cu etichete deasupra, deci mosteneste viteza aceasta. La date mici nu simti diferenta. La un milion de randuri, o bucla scrisa de mana devine problema.

## 6. Prima privire: `shape`, `info`, `describe`

Trei comenzi pe care le rulezi mereu, in ordinea asta:

- `shape` = (randuri, coloane)
- `info()` = tipul fiecarei coloane si cate valori **nenule** are
- `describe()` = numar, medie, deviatie (cu `ddof=1`), minim, cuartile, maxim, pentru coloanele numerice

In [ ]:
print(elevi.shape)
elevi.info()

**Prezice:** uita-te la `describe()` de mai jos. Ce numere ti se par suspecte? (Cauta minimul, maximul, si `count` fata de 245 de randuri.)

<details><summary>Raspuns</summary>

- `varsta`: minim 2, maxim 150. Un elev de liceu nu are 2 sau 150 de ani.
- `absente`: maxim 120, dar 75% dintre elevi au cel mult 8. Ceva nu se leaga.
- `nota_info`, `ore_studiu`: `count` mai mic decat 245 inseamna valori lipsa.
- `id`: maxim 240, dar sunt 245 de randuri. Cineva apare de doua ori.
</details>

In [ ]:
elevi.describe()

## 7. Selectii: coloane, randuri, masti

```python
elevi['nota_info']                 # o coloana -> Series
elevi[['nota_info', 'absente']]    # mai multe coloane -> DataFrame (lista in interiorul parantezelor!)
elevi.loc[5, 'nota_info']          # dupa ETICHETA: randul 5, coloana nota_info
elevi.iloc[0:3, 0:2]               # dupa POZITIE, ca la liste (capatul din dreapta nu intra)
```
Diferenta care musca: `.loc[0:3]` **include** capatul 3, `.iloc[0:3]` **nu**.

Partea cea mai folosita e filtrarea cu o **masca**: un sir de `True`/`False`, cate unul pe rand. Intai o faci de mana, pe liste:

In [ ]:
%%vezi
nota = [4.5, 8.2, 9.1, 6.0, 8.8]
absente = [2, 9, 1, 3, 2]
masca = []
for n, a in zip(nota, absente):
    masca.append(n >= 8 and a <= 3)             #: True doar daca ambele conditii sunt adevarate
bune = [n for n, m in zip(nota, masca) if m]    #: pastram doar pozitiile unde masca e True

In pandas masca se construieste dintr-o singura expresie, pe toata coloana deodata, si se pune in paranteze drepte. Aici nu mai poti scrie `and` / `or` / `not`: pentru mastile pandas si NumPy ai `&`, `|`, `~`, **iar fiecare conditie trebuie pusa in paranteze**. Uite de ce.

In [ ]:
%%vezi
import numpy as np
n = np.array([4.5, 8.2, 9.1, 6.0])
a = np.array([2, 9, 1, 3])
m1 = n >= 8            #: o masca: True / False pe fiecare pozitie
m2 = a <= 3
m = m1 & m2            #: & compara element cu element: merge
m = m1 and m2          #: and vrea UN singur adevarat/fals pentru tot array-ul: eroare

In [ ]:
%%vezi
import numpy as np
n = np.array([4.5, 8.2, 9.1, 6.0])
a = np.array([2, 9, 1, 3])
gresit = n >= 8 & a <= 3      #: & are prioritate mai mare decat >= si <=, deci se calculeaza intai 8 & a

Ordinea operatiilor e cauza: `n >= 8 & a <= 3` se citeste `n >= (8 & a) <= 3`, fara paranteze. Eroarea ("truth value ... is ambiguous") vorbeste despre `and`, dar vinovatul adevarat e lipsa parantezelor. Varianta corecta:

In [ ]:
buni = elevi[(elevi['nota_info'] >= 7) & (elevi['absente'] <= 5)]
print(len(buni), 'elevi cu nota_info >= 7 si cel mult 5 absente')

# sau, cu | : clasa a 9-a SAU a 12-a
print(len(elevi[(elevi['clasa'] == 9) | (elevi['clasa'] == 12)]))
# isin face la fel de scurt
print(elevi['clasa'].isin([9, 12]).sum())

## 8. Sortare, frecvente, grupari

`sort_values` ordoneaza. `value_counts` numara cate ori apare fiecare valoare. Numaratul de mana e un dictionar:

In [ ]:
%%vezi
raspunsuri = ['da', 'nu', 'Da', 'nu', 'da', 'DA ', 'nu']
frecv = {}
for r in raspunsuri:
    if r in frecv:
        frecv[r] = frecv[r] + 1     #: cheia exista deja: crestem contorul
    else:
        frecv[r] = 1                #: prima aparitie

Observa ca `'da'`, `'Da'` si `'DA '` sunt **trei chei diferite**. Pentru calculator nu sunt acelasi raspuns. Exact asta e problema din coloana `meditatii`:

In [ ]:
print(elevi['meditatii'].value_counts(dropna=False))
elevi.sort_values('absente', ascending=False).head(4)

**`groupby`** = imparte randurile in grupuri dupa o coloana, calculeaza ceva pe fiecare grup, aduna rezultatele (split / apply / combine). De mana, cu un dictionar de liste:

In [ ]:
%%vezi
clasa = [9, 10, 9, 11, 10, 9]
nota = [6.0, 8.0, 7.0, 5.0, 9.0, 8.0]
grupuri = {}
for cl, n in zip(clasa, nota):
    if cl not in grupuri:
        grupuri[cl] = []
    grupuri[cl].append(n)                  #: fiecare nota intra in lista clasei ei
medii = {}
for cl in grupuri:
    medii[cl] = sum(grupuri[cl]) / len(grupuri[cl])   #: apoi facem media pe fiecare grup

In [ ]:
print(elevi.groupby('clasa')['nota_info'].mean())
elevi.groupby('clasa').agg(elevi=('id', 'count'), nota_medie=('nota_info', 'mean'), ore_mediana=('ore_studiu', 'median'))

## 9. Distributii: histograma

O histograma imparte valorile in "cosuri" (bare) si numara cate cad in fiecare. Un singur parametru conteaza mult: **numarul de bare**. Uita-te cum se schimba imaginea pentru `nota_info`, de la 2 la 40 de bare:

In [ ]:
anim_histograma(elevi['nota_info'], eticheta='nota_info')

Prea putine bare ascund forma, prea multe o umplu de zgomot. Punct de plecare: aproximativ radical din `n` bare (formula lui Sturges, `log2(n) + 1`, da un numar apropiat).

Doua lucruri de citit dintr-o histograma: **forma** (un varf? doua?) si **asimetria** (coada mai lunga intr-o parte). Daca coada e in dreapta, media > mediana. `ore_studiu` e asa:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, col in zip(ax, ['nota_info', 'ore_studiu']):
    x = elevi[col].dropna()
    a.hist(x, bins=20, color='#2563eb', edgecolor='white')
    a.axvline(x.mean(), color='#ea580c', lw=2, label=f'media {x.mean():.2f}')
    a.axvline(x.median(), color='#15803d', lw=2, ls='--', label=f'mediana {x.median():.2f}')
    a.set_title(f'{col}: asimetrie (skew) = {x.skew():.2f}')
    a.legend()
plt.tight_layout()
plt.show()

`skew()` aproape de 0 = simetric, pozitiv = coada in dreapta, negativ = coada in stanga. `ore_studiu` (skew ~0.9) are coada in dreapta: cativa elevi invata mult mai mult decat restul.

## 10. Outlieri

Un outlier e o valoare care iese din tiparul restului. Prima intrebare nu e "cum il sterg", ci **"de ce e acolo?"**: greseala de tastare (varsta 150), sau caz real rar (o casa de 2,6 milioane)?

Ai regula IQR de mana (sectiunea 4). Aplic-o pe `absente` cu pandas si uita-te si la o **boxplot**: cutia e intre `Q1` si `Q3`, linia e mediana, mustatile ajung la ultimele valori din garduri, iar punctele sunt outlierii.

In [ ]:
a = elevi['absente']
q1, q3 = a.quantile(0.25), a.quantile(0.75)
iqr = q3 - q1
jos, sus = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1}, Q3={q3}, IQR={iqr}, garduri: [{jos}, {sus}]')
print('outlieri:', sorted(a[(a < jos) | (a > sus)].tolist()))

fig, ax = plt.subplots(figsize=(4, 3.2))
ax.boxplot(a)
ax.set_title('absente')
plt.show()

Regula IQR prinde 15 (un elev cu multe absente, dar plauzibil) **si** pe 87 si 120 (aproape sigur greseli). Regula e un semnal de alarma, nu un verdict. Ce faci apoi depinde de context: corectezi daca stii valoarea, marchezi ca lipsa, limitezi la un maxim, sau pastrezi.

Ce face un outlier cu media? Animatia adauga o valoare care urca pas cu pas, peste primele 20 de note la matematica. Urmareste cele doua linii:

In [ ]:
anim_medie_mediana(elevi['nota_mate'].head(20), eticheta='nota_mate')

Media urmareste valoarea extrema, mediana abia se misca. Cu 245 de valori, un singur outlier misca media mult mai putin (il imparti la 245), dar in coloana `absente` sunt doua:

In [ ]:
a = elevi['absente']
print('cu tot cu outlieri :', round(a.mean(), 2), 'media |', a.median(), 'mediana')
sane = a[a < 50]
print('fara cele 2 extreme:', round(sane.mean(), 2), 'media |', sane.median(), 'mediana')

## 11. Relatii intre coloane: corelatie

Coeficientul de corelatie Pearson `r` (intre -1 si 1) masoara cat de bine se aliniaza doua coloane pe o **dreapta**: +1 = dreapta crescatoare perfecta, 0 = nicio aliniere liniara, -1 = descrescatoare. Formula de mana:

`r = suma((x - mx) * (y - my)) / radical(suma((x - mx)^2) * suma((y - my)^2))`

In [ ]:
%%vezi
x = [1, 2, 3, 4, 5]
y = [2, 4, 5, 4, 5]
n = len(x)
mx = sum(x) / n
my = sum(y) / n
sxy = 0
sxx = 0
syy = 0
for a, b in zip(x, y):
    sxy += (a - mx) * (b - my)     #: cat variaza impreuna x si y
    sxx += (a - mx) ** 2
    syy += (b - my) ** 2
r = sxy / (sxx * syy) ** 0.5       #: normalizam, ca sa iasa intre -1 si 1

In [ ]:
print(np.corrcoef([1, 2, 3, 4, 5], [2, 4, 5, 4, 5])[0, 1])
cols = ['nota_mate', 'nota_info', 'ore_studiu', 'absente', 'calificat']
cm = elevi[cols].corr()          # NaN-urile sunt sarite pereche cu pereche
fig, ax = plt.subplots(figsize=(5.5, 4.2))
im = ax.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha='right')
ax.set_yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im)
plt.tight_layout()
plt.show()

Ce vezi: `nota_info` si `nota_mate` merg impreuna (~0.5), `ore_studiu` si `nota_info` mai slab (~0.33), iar `absente` e legat foarte slab de orice. Un scatter plot arata mai mult decat un numar:

In [ ]:
d = elevi.dropna(subset=['nota_info'])
plt.figure(figsize=(6, 3.8))
plt.scatter(d['nota_mate'], d['nota_info'], c=d['calificat'], cmap='coolwarm', s=18, alpha=0.8)
plt.xlabel('nota_mate')
plt.ylabel('nota_info')
plt.title('rosu = calificat, albastru = necalificat')
plt.show()

Un singur punct extrem poate strica corelatia. Animatia muta un punct in coltul de jos-dreapta al norului tau:

In [ ]:
anim_corelatie(elevi['nota_mate'], elevi['nota_info'], eticheta_x='nota_mate', eticheta_y='nota_info')

**Corelatia nu inseamna cauzalitate.** Ai vazut in prefata cursului de ce: doua lucruri pot merge impreuna pentru ca un al treilea le influenteaza pe amandoua, sau pur si simplu din intamplare. Aici, `ore_studiu` creste si nota la info, si sansa de calificare, deci si notele si calificarea "se misca" impreuna fara ca una s-o cauzeze neaparat pe cealalta.

Din intamplare? Doua miscari complet independente (nu au nimic de-a face una cu alta) pot iesi corelate puternic:

In [ ]:
rng = np.random.default_rng(7)
a = rng.normal(size=300).cumsum()      # un "drum" aleator
b = rng.normal(size=300).cumsum()      # alt drum, complet independent
print('corelatia dintre doua drumuri independente:', round(np.corrcoef(a, b)[0, 1], 2))
plt.figure(figsize=(6, 2.6))
plt.plot(a)
plt.plot(b)
plt.show()

## 12. Coloane categorice si tinta

Pentru o coloana categorica te intereseaza: cate categorii sunt, cat de des apare fiecare, si **cum se leaga de tinta**. Iar despre tinta: e echilibrata? Daca 95% dintre randuri au aceeasi clasa, un model care prezice mereu "da" are 95% acuratete si e inutil.

In [ ]:
print(elevi['calificat'].value_counts(normalize=True).round(3))
curat = elevi['meditatii'].str.strip().str.lower()    # strip taie spatiile de la capete, lower face literele mici
print(curat.value_counts(normalize=True).round(3))
print(elevi.groupby(curat)['calificat'].agg(['count', 'mean']))

33% dintre elevi sunt calificati: tinta e dezechilibrata, dar nu extrem. Iar `meditatii` nu pare sa schimbe mult rata de calificare (35% la "da", 31% la "nu"): diferenta e mica pe cate 77 si 155 de elevi si ar putea fi intamplare.

## 13. Checklist-ul EDA

Aceeasi lista pentru orice set de date. Nu o memora, tine-o in fata:

| # | intrebare | cum afli |
|---|---|---|
| 1 | Ce e un rand? Cate sunt? | `df.shape`, `df.head()` |
| 2 | Ce tip are fiecare coloana? Sunt tipurile corecte? | `df.info()` |
| 3 | Care e tinta? E echilibrata? | `df['tinta'].value_counts(normalize=True)` |
| 4 | Ce lipseste? Cat? | `df.isna().sum()` |
| 5 | Sunt randuri duplicate? | `df.duplicated().sum()` |
| 6 | Au sens intervalele? (minim, maxim) | `df.describe()` |
| 7 | Sunt categoriile scrise consecvent? | `df['c'].value_counts()` |
| 8 | Cum arata distributiile? | `df['c'].hist()` |
| 9 | Sunt outlieri? | regula IQR, boxplot |
| 10 | Ce relatii exista? | `df.corr()`, scatter |

Le aplicam pe `elevi.csv`, pe rand. Intrebarile 1, 2 si 3 le-ai facut deja (245 de randuri, un rand = un elev, tinta = `calificat`, tipurile sunt in regula, cu `meditatii` ca text).

**Intrebarea 4: ce lipseste?**

In [ ]:
lipsa = elevi.isna().sum()
print(pd.DataFrame({'lipsa': lipsa, 'procent': (lipsa / len(elevi) * 100).round(1)}))
print('randuri cu cel putin o valoare lipsa:', elevi.isna().any(axis=1).sum())
print('rata de lipsa la ore_studiu, pe clase:')
print(elevi.groupby('clasa')['ore_studiu'].apply(lambda s: s.isna().mean().round(3)))

Trei coloane au goluri. Iar `ore_studiu` lipseste mult mai des la clasa a 12-a (~29%) decat la celelalte (2-7%). Golurile nu sunt aleatoare. Ce faci cu ele e subiectul notebook-ului urmator.

**Intrebarea 5: duplicate?** **6: intervale?**

In [ ]:
print('randuri duplicate exact:', elevi.duplicated().sum())
print(elevi[elevi.duplicated(keep=False)].sort_values('id').head(4))

print('varste suspecte:')
print(elevi.loc[(elevi['varsta'] < 12) | (elevi['varsta'] > 20), ['id', 'clasa', 'varsta']])

Cinci randuri sunt copii exacte (aceeasi persoana, aceleasi valori, acelasi `id`), iar doua varste sunt imposibile: 150 (la un elev de clasa a 9-a) si 2 (la clasa a 11-a). Ai grija la comparatii: pe `NaN`, orice comparatie da `False`, deci mastile de mai sus nu le prind.

**Intrebarea 7: categorii.** Am vazut-o deja mai sus: `meditatii` are `da`, `Da`, `DA `, `nu`, ` nu`. Reparatia e `str.strip().str.lower()`.

**Intrebarile 8-10** (distributii, outlieri, relatii) le-am rezolvat la sectiunile 9-11: `absente` are 2 valori extreme (87, 120), `ore_studiu` e asimetrica la dreapta.

Rezultatul unei sesiuni de EDA e o **lista de probleme**, apoi o functie de curatare:

In [ ]:
curat = elevi.drop_duplicates().copy()
curat['meditatii'] = curat['meditatii'].str.strip().str.lower()
curat.loc[(curat['varsta'] < 12) | (curat['varsta'] > 20), 'varsta'] = np.nan

print(elevi.shape, '->', curat.shape)
print(curat['meditatii'].value_counts(dropna=False).to_dict())
print(curat['varsta'].describe().round(2).to_dict())

Am lasat `absente` neatinsa (decizia o iei cu cine a adus datele) si am lasat golurile: le tratam la topicul urmator. Vezi ca nu am modificat `elevi` original, ci o copie: pastrezi mereu datele brute.

### Aceeasi lista, pe `case.csv`

300 de apartamente, tinta = `pret_eur`. Aplicam checklist-ul pe scurt.

In [ ]:
case = pd.read_csv('date/case.csv')
print(case.shape, '| duplicate:', case.duplicated().sum())
print(case.isna().sum()[lambda s: s > 0].to_dict())
case.describe().round(1)

Prima diferenta fata de `elevi`: **scarile**. `pret_eur` e de ordinul sutelor de mii, `suprafata_mp` de ordinul zecilor, `camere` intre 1 si 6. Multe modele (KNN, regresie cu regularizare, retele) se vor lasa dominate de coloana cu numerele cele mai mari, de aceea vei invata scalarea. Iar `pret_eur` are maximul la 2,6 milioane cand mediana e 76.000: trei preturi extreme.

In [ ]:
p = case['pret_eur']
print('media:', round(p.mean()), '| mediana:', p.median(), '| skew:', round(p.skew(), 1))
ok = case[case['pret_eur'] < 500_000]
print('corelatie suprafata-pret cu extremele  :', round(case['suprafata_mp'].corr(case['pret_eur']), 2))
print('corelatie suprafata-pret fara extreme  :', round(ok['suprafata_mp'].corr(ok['pret_eur']), 2))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(p, bins=40, color='#2563eb', edgecolor='white')
ax[0].set_title('pret_eur: coada foarte lunga')
ax[1].scatter(case['suprafata_mp'], p, s=10, alpha=0.6)
ax[1].set_xlabel('suprafata_mp')
ax[1].set_ylabel('pret_eur')
plt.tight_layout()
plt.show()
print(case.groupby('oras')['pret_eur'].agg(['count', 'mean', 'median']).round(0))

Trei preturi extreme coboara corelatia de la ~0.90 la ~0.27. Iar la Bucuresti media (135.000) e mult peste mediana (80.000), exact din cauza lor. **Asta e motivul pentru care in date cu extreme te uiti la mediana si la corelatia calculata cu si fara ele.** Aceleasi preturi sunt outlieri reali sau greseli? De aici incolo e o decizie, nu un calcul.

## 14. Capcane

**Capcana 1: media ignora `NaN` in tacere.** `mean()` nu da eroare pe o coloana cu goluri, o face pe cele existente. Iar `len` numara si golurile.

In [ ]:
x = elevi['nota_info']
print('len:', len(x), '| count:', x.count(), '| lipsa:', x.isna().sum())
print('media pe', x.count(), 'valori:', round(x.mean(), 3))

**Capcana 2: modificarea in lant nu modifica nimic.** `df[masca]['col'] = ...` lucreaza pe o copie temporara. In pandas 2 primesti un avertisment, in pandas 3 nici macar asta. Foloseste `.loc[masca, 'col']`.

In [ ]:
import warnings
d = elevi.copy()
print('absente maxime la clasa a 12-a, inainte:', d.loc[d['clasa'] == 12, 'absente'].max())
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    d[d['clasa'] == 12]['absente'] = 0           # pare ca merge... dar nu
print('dupa atribuirea in lant                :', d.loc[d['clasa'] == 12, 'absente'].max())

d.loc[d['clasa'] == 12, 'absente'] = 0           # asa da
print('dupa .loc                              :', d.loc[d['clasa'] == 12, 'absente'].max())

**Capcana 3: corelatia zero nu inseamna "nicio relatie".** `r` masoara doar aliniere pe dreapta. O relatie perfecta, dar curba, poate da `r = 0`:

In [ ]:
x = np.linspace(-3, 3, 61)
y = x ** 2
print('r =', round(np.corrcoef(x, y)[0, 1], 3), '(dar y depinde total de x)')
plt.figure(figsize=(4, 2.6))
plt.scatter(x, y, s=8)
plt.show()

**Capcana 4: o coloana numerica citata ca text.** Daca intr-o coloana apare un singur `'n/a'` sau `'fara nota'`, pandas o citeste intreaga ca text (`object`), iar `mean()` nu mai merge cum crezi. `info()` te prinde (tipul nu e `float64`). `pd.to_numeric(errors='coerce')` transforma ce nu e numar in `NaN`.

In [ ]:
rating = pd.Series(['4', '5', 'fara nota', '3'])
print(rating.dtype)
nr = pd.to_numeric(rating, errors='coerce')
print(nr.tolist(), '| media:', nr.mean())

**Capcana 5: `id` (si orice numar de ordine) nu e feature.** Corelatia lui cu ceva poate iesi mica, dar un arbore de decizie poate imparti dupa `id` daca datele sunt sortate dupa tinta. Il scoti inainte de model.

### Cum citesti erorile de pandas

Aceleasi doua reguli: ultima linie spune *ce* s-a intamplat, numarul liniei spune *unde*.

In [ ]:
%%vezi
import pandas as pd
df = pd.DataFrame({'nota': [7, 9, 5], 'absente': [1, 4, 0]})
x = df['nota']
y = df['nota_info']     #: KeyError = numele coloanei nu exista (greseala de scriere?)

| eroare | de obicei inseamna |
|---|---|
| `KeyError: 'nota_info'` | coloana nu exista: verifica `df.columns` (spatii, litere mari?) |
| `ValueError: The truth value of a Series is ambiguous` | ai folosit `and` / `or` / `not` pe o coloana. Foloseste `&`, `|`, `~` |
| `TypeError: '>' not supported between 'str' and 'int'` | coloana e text (`object`) si o compari cu un numar: `info()` |
| `AttributeError: Can only use .str accessor with string values` | ai apelat `.str` pe o coloana numerica |
| `SettingWithCopyWarning` | modifici o copie: foloseste `.loc` sau `.copy()` |

## Exercitii

Pe rand. Fiecare are o celula de completat si una de verificare. Ai deja incarcat `elevi` (cu tot cu problemele lui, nu l-am curatat). Daca nu stii o functie, cauta-o in documentatie: la olimpiada asa faci.

**Exercitiul 1.** Media si mediana coloanei `nota_mate`.

In [ ]:
media_mate = None      # TODO
mediana_mate = None    # TODO

In [ ]:
verifica_egal('media', media_mate, 5.886775510204082)
verifica_egal('mediana', mediana_mate, 5.91)

**Exercitiul 2.** Varianta de mana, cu un parametru `ddof` (0 = populatie, 1 = esantion). Fara `np.var`, fara `statistics`. O bucla si formula din sectiunea 3.

In [ ]:
def varianta(x, ddof):
    pass  # TODO

In [ ]:
import random
verifica('varianta', varianta, [(([2, 4, 4, 4, 5, 5, 7, 9], 0), 4.0), (([2, 4, 4, 4, 5, 5, 7, 9], 1), 32 / 7)],
         ascunse=[(([5, 5, 5], 0), 0.0), (([1, 2], 1), 0.5)],
         referinta=lambda x, d: float(np.var(x, ddof=d)),
         generator=lambda: ([round(random.uniform(0, 20), 1) for _ in range(random.randint(3, 12))], random.choice([0, 1])),
         indiciu='media, apoi suma patratelor abaterilor, apoi impartirea la len(x) - ddof')

**Exercitiul 3.** Gardurile IQR de mana. Intoarce perechea `(gard_jos, gard_sus)`. Cuantilele le calculezi tu, cu interpolare liniara (sectiunea 4), nu cu `np.percentile`.

In [ ]:
def garduri_iqr(valori):
    s = sorted(valori)
    pass  # TODO: q1, q3 cu interpolare; apoi gardurile

In [ ]:
import random


def garduri_ref(v):
    q1, q3 = np.percentile(v, [25, 75])
    return float(q1 - 1.5 * (q3 - q1)), float(q3 + 1.5 * (q3 - q1))


verifica('garduri_iqr', garduri_iqr, [(([2, 3, 4, 6, 8, 9, 10, 40],), (-4.5, 17.5)), (([1, 2, 3, 4, 5],), (-1.0, 7.0))],
         ascunse=[(([7],), (7.0, 7.0))],
         referinta=garduri_ref, generator=lambda: ([random.randint(0, 50) for _ in range(random.randint(4, 30))],),
         indiciu='pozitia lui Q1 este 0.25 * (n - 1) in lista sortata')

**Exercitiul 4.** Doua intrebari din checklist, cu pandas: cate randuri sunt duplicate exact in `elevi` (`nr_duplicate`) si ce **procent** din `ore_studiu` lipseste (`procent_lipsa_ore`, un numar intre 0 si 100).

In [ ]:
nr_duplicate = None        # TODO
procent_lipsa_ore = None   # TODO

In [ ]:
verifica_egal('duplicate', nr_duplicate, 5)
verifica_egal('procent lipsa', procent_lipsa_ore, 9.795918367346939)

**Exercitiul 5.** Curata `meditatii` (spatii si litere mari), apoi numara cati dintre elevii cu meditatii (`'da'`) au `calificat == 1`. Atentie la duplicate: aici le lasam in pace, numaram pe randurile din `elevi` asa cum sunt.

In [ ]:
calificati_cu_meditatii = None   # TODO

In [ ]:
verifica_egal('calificati cu meditatii', calificati_cu_meditatii, 27,
               indiciu='str.strip().str.lower(), apoi doua conditii cu & si paranteze')

**Exercitiul 6.** Corelatia Pearson de mana, pe doua liste. Nu folosi `np.corrcoef`.

In [ ]:
def pearson(x, y):
    pass  # TODO

In [ ]:
import random
verifica('pearson', pearson, [(([1, 2, 3, 4, 5], [2, 4, 6, 8, 10]), 1.0), (([1, 2, 3], [3, 2, 1]), -1.0),
                              (([1, 2, 3, 4, 5], [2, 4, 5, 4, 5]), 0.7745966692414834)],
         referinta=lambda x, y: float(np.corrcoef(x, y)[0, 1]),
         generator=lambda: tuple(zip(*[(random.randint(0, 30), random.randint(0, 30)) for _ in range(random.randint(4, 15))])),
         indiciu='formula din sectiunea 11: suma produselor abaterilor / radical din produsul sumelor de patrate')

**Exercitiul 7.** Nota medie la informatica pe fiecare clasa, ca dictionar `{clasa: media}`. Ai nevoie de `groupby`, `mean` si `to_dict()`.

In [ ]:
medie_pe_clasa = None   # TODO

In [ ]:
verifica_egal('medie pe clasa', medie_pe_clasa, {9: 5.8475, 10: 5.854, 11: 5.660727272727272, 12: 5.527777777777778}, indiciu="groupby('clasa')['nota_info'].mean().to_dict()")

**Exercitiul 8.** Cati elevi au `nota_info >= 7` **si** cel mult 5 absente? Fii atent la paranteze.

In [ ]:
nr_buni = None   # TODO

In [ ]:
verifica_egal('nr_buni', nr_buni, 12)
scor()

## Recapitulare

- EDA = intrebari puse datelor **inainte** de model. Aceeasi lista pentru orice set (checklist-ul din sectiunea 13).
- Tipuri de coloane: numeric, categoric, ordinal, tinta, identificator (nu e feature).
- De mana: media (`suma / n`), mediana (sortezi, iei mijlocul), varianta (`/ n` populatie, `/ (n - 1)` esantion; NumPy `ddof=0`, pandas `ddof=1`), cuantile prin interpolare liniara, IQR si gardurile `Q1 - 1.5 IQR`, `Q3 + 1.5 IQR`.
- Pandas/NumPy sunt mai rapide pentru ca bucla ruleaza in C pe memorie continua. Tu trebuie sa stii ce calculeaza.
- Masti: `&`, `|`, `~`, cu **paranteze** la fiecare conditie. Modificari: `.loc[masca, 'col'] = ...`, nu in lant.
- Outlierii: intreaba de ce sunt acolo. Media e trasa de ei, mediana nu.
- Corelatia masoara doar aliniere liniara, e sensibila la outlieri si nu e cauzalitate.
- Rezultatul EDA e o lista de probleme din date. Pe `elevi.csv`: valori lipsa (nu aleatoare), absente extreme, varsta imposibila, `meditatii` scris inconsistent, 5 duplicate.

Legatura cu ONIA: la orice sarcina cu un CSV, primele 10 minute sunt EDA. Datele murdare pierd mai multe puncte decat alegerea modelului. Tema (notebook-ul urmator) are un raport EDA complet, cu verificari automate.